## Tablemate - Cell 1 - Importer och Table

In [ ]:
import csv
from datetime import datetime, timedelta
import os
from openai import OpenAI, APIError

class Table: #skapar en klass Table som representerar en tabell för borden
    def __init__(
        self,
        table_id,
        table_capacity,
        table_placement,
        table_type,
        table_description,
        is_bookable):

        self.table_id = table_id
        self.table_capacity = table_capacity
        self.table_placement = table_placement
        self.table_type = table_type
        self.table_description = table_description
        self.is_bookable = is_bookable

    def display_information(self): #skapar en metod display_information som skriver ut information om tabellen
        return (
        f" Table: {self.table_id}, "
        f" Seats: {self.table_capacity}, "
        f" Placement: {self.table_placement}, "
        f" Type: {self.table_type}, "
        f" Description: {self.table_description}, "
        f" Bookable: {self.is_bookable}")

    def can_seat_guests(self, guest_count):
        return self.is_bookable and guest_count <= self.table_capacity

### Cell 2 - TerraceTable

In [ ]:
class TerraceTable(Table): #skapar en barnklass med TerraceTable
    def display_information(self):
        base_information = super().display_information()

        return (
            f"{base_information}, "
            f"Outdoor Seating: Yes" )

### Cell 3 - load_tables()

In [ ]:
def load_tables(file_import): #skapar en funktion --load_tables-- som läser in tabeller från en CSV-fil
    tables = [] #skapar listan tables som kommer att innehålla alla tabeller

    try:
        with open(file_import, "r", encoding="utf-8", newline="") as file: #öppnar datafilen
            reader = csv.DictReader(file) #använder rubriker som nycklar för att läsa in data
            for row in reader: #loopar igenom ett bord i taget

                row["table_id"] = int(row["table_id"]) #konverterar till en int
                row["table_capacity"] = int(row["table_capacity"]) #konverterar  till en int
                row["is_bookable"] = row["is_bookable"].strip().lower() in ("true", "yes") #konverterar  till en bool

                if row["table_type"].strip().lower() == "terrace": #
                    table = TerraceTable(
                        row["table_id"],
                        row["table_capacity"],
                        row["table_placement"],
                        row["table_type"],
                        row["table_description"],
                        row["is_bookable"])
               
                else:
                    table = Table(
                        row["table_id"],
                        row["table_capacity"],
                         row["table_placement"],
                        row["table_type"],
                        row["table_description"],
                        row["is_bookable"])

                tables.append(table) #Alla bord läggs till oavsett typ.

        return tables

    except FileNotFoundError: #om filen inte hittas
        print(f"Error: The file '{file_import}' was not found.") #skriver ut ett felmeddelande

    except (ValueError, KeyError) as error: #om det finns ett värdefel eller nyckelfel
        print(f"Error while reading '{file_import}': {error}") #skriver ut ett felmeddelande

    except csv.Error as error: #om det finns ett csv-fel
        print(f"Error: An error occurred while processing '{file_import}': {error}") #skriver ut ett felmeddelande
    return []

### Cell 4 - load_bookings()

In [ ]:
def load_bookings(file_import):
    bookings = []

    try:
        with open(file_import,"r",encoding="utf-8", newline="") as file:
            reader = csv.DictReader(file)

            for row in reader:
                row["booking_id"] = int(row["booking_id"])
                row["guest_count"] = int(row["guest_count"])
                row["table_id"] = int(row["table_id"])

                bookings.append(row)

        return bookings

    except FileNotFoundError:
        print(f"Error: The file '{file_import}' was not found.") #skriver ut ett felmeddelande

    except (ValueError, KeyError) as error: #om det finns ett värdefel eller nyckelfel
        print(f"Error while reading '{file_import}': {error}") #skriver ut ett felmeddelande

    except csv.Error as error: #om det finns ett csv-fel
        print(f"Error: An error occurred while processing '{file_import}': {error}") #skriver ut ett felmeddelande

    return []

### Cell 5 - is_table_available()

In [ ]:
def is_table_available(
        bookings,
        table_id,
        booking_date,
        booking_time):

    for booking in bookings:
        same_table = booking["table_id"] == table_id
        same_date = booking["booking_date"] == booking_date
        same_time = booking["booking_time"] == booking_time
        is_active = (booking["booking_status"].lower() == "active")

        if (
            same_table
            and same_date
            and same_time
            and is_active):

            return False

    return True

### Cell 6 - bokningsfält och save_booking()

In [ ]:
BOOKING_FIELDS = [
    "booking_id",
    "customer_name",
    "booking_date",
    "booking_time",
    "guest_count",
    "table_id",
    "booking_status",
    "created_at",]

def save_booking(file_import, booking):
    try:
        with open(file_import, "a", encoding="utf-8", newline="") as file:
            writer = csv.DictWriter(file,fieldnames=BOOKING_FIELDS)

            if file.tell() == 0:
                writer.writeheader()
            writer.writerow(booking)

        return True

    except (OSError, csv.Error) as error:
        print(f"Error while saving the booking: {error}")

        return False

### Cell 7 - create_booking()

In [ ]:
def create_booking(
        bookings,
        table,
        customer_name,
        booking_date,
        booking_time,
        guest_count,
        file_import):

        if not table.can_seat_guests(guest_count):
            print( "The table cannot accommodate the requested amount of guests.")

            return None

        if not is_table_available(
              bookings,
              table.table_id,
              booking_date,
              booking_time):
            print("The table is already booked at the selected time and date.")

            return None
        
        new_booking_id = 1

        for existing_booking in bookings:
             if (existing_booking["booking_id"] >= new_booking_id):
  
                  new_booking_id = (existing_booking["booking_id"] + 1)

        new_booking = {
             "booking_id": new_booking_id,
             "customer_name": customer_name,
             "booking_date": booking_date,
             "booking_time": booking_time,
             "guest_count": guest_count,
             "table_id": table.table_id,
             "booking_status": "active",
             "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

        if save_booking(file_import, new_booking):
             bookings.append(new_booking)

             print(f"Booking {new_booking_id} was created for table {table.table_id}." )

             return new_booking

        return None

### Cell 8 - save_all_bookings()

In [ ]:
def save_all_bookings(file_import, bookings):
    try:
        with open(file_import, "w", encoding="utf-8", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=BOOKING_FIELDS)

            writer.writeheader()
            writer.writerows(bookings)

        return True

    except (OSError, csv.Error) as error:
        print(f"Error while updating bookings: {error}")

        return False

### Cell 9 - cancel_booking()

In [ ]:
def cancel_booking(bookings, booking_id,file_import):

    for booking in bookings:
        correct_id = (
            booking["booking_id"]) == booking_id

        is_active = (
            booking["booking_status"].strip().lower() == "active")

        if correct_id and is_active:
            booking["booking_status"] = "cancelled"

            if save_all_bookings(file_import, bookings):
                print(f"Booking {booking_id} was cancelled.")

                return True

            booking["booking_status"] = "active"

            return False

    print(f"No active booking with ID {booking_id} was found.")

    return False

### Cell 10 - find_available_tables()

In [ ]:
def find_available_tables(tables, bookings, guest_count, booking_date, booking_time):

    available_tables = []

    for table in tables:
        has_capacity = table.can_seat_guests(guest_count)

        is_available = is_table_available(
            bookings,
            table.table_id,
            booking_date,
            booking_time)

        if has_capacity and is_available:
            available_tables.append(table)

    return available_tables

### Cell 11 - recommend_table_with_grok()

In [ ]:
def recommend_table_with_grok(available_tables, customer_preferences):

    if not available_tables:
        print("There are no available tables to recommend.")
        return None

    if not customer_preferences.strip():
        print("Please enter your table preferences.")
        return None

    api_key = os.getenv("XAI_API_KEY")

    if not api_key:
        print("The enviroment variable XAI_API_KEY is missing.")
        return None

    table_information = "\n".join(
        (
            f"Table ID: {table.table_id}, "
            f"Seats: {table.table_capacity}, "
            f"Placement: {table.table_placement}, "
            f"Type: {table.table_type}, "
            f"Description: {table.table_description}, "
        )
            
        for table in available_tables)

    try:
        client = OpenAI(
            api_key=api_key,
            base_url="https://api.openai.com/v1")

        response = client.responses.create(
            model="grok-4.7", 
            store=False, 
            input=[{
            "role": "user", "content":
            ("You are a restaurant booking assistant. "
            "Recommend exactly one table from the provided list. "
            "Never recommend a table that is not in the list. "
            "Give the table ID and a short explantation.")},

            {"role": "user", "content": (
                f"Customer perferences: {customer_preferences}\n\n"
                f"Available tables:\n{table_information}"
                ) 
            }
        ]
    )

        return response.output_text.strip()

    except APIError as error:
        print(f"An error occurred while communicating with the OpenAI API: {error}")
        return None 

### Cell 12 - Validate bookings //new date and time

In [ ]:
def select_booking_date(days_to_show=14):
    available_dates = []

    for day_number in range(days_to_show):
        date_option = datetime.now().date() + timedelta(days=day_number)
        available_dates.append(date_option)

    print("\nAvailable dates:")

    for number, date_option in enumerate(available_dates, start=1):
        print(
            f"{number}. "
            f"{date_option.strftime('%A, %Y-%m-%d')}"
        )

    while True:
        try:
            selected_number = int(
                input("Choose a date by entering its number: ")
            )

            if 1 <= selected_number <= len(available_dates):
                selected_date = available_dates[selected_number - 1]
                return selected_date.strftime("%Y-%m-%d")

            print("Please choose a number from the list.")

        except ValueError:
            print("Please enter a number.")


def select_booking_time():
    available_times = [
        "17:00",
        "18:00",
        "19:00",
        "20:00",
        "21:00"
    ]

    print("\nAvailable times:")

    for number, time_option in enumerate(available_times, start=1):
        print(f"{number}. {time_option}")

    while True:
        try:
            selected_number = int(
                input("Choose a time by entering its number: ")
            )

            if 1 <= selected_number <= len(available_times):
                return available_times[selected_number - 1]

            print("Please choose a number from the list.")

        except ValueError:
            print("Please enter a number.")

### display booking info//ny

In [ ]:
def display_booking_confirmation(booking):
    print("\n================================")
    print("      BOOKING CONFIRMED")
    print("================================")
    print(f"Booking ID: {booking['booking_id']}")
    print(f"Customer: {booking['customer_name']}")
    print(f"Date: {booking['booking_date']}")
    print(f"Time: {booking['booking_time']}")
    print(f"Guests: {booking['guest_count']}")
    print(f"Table: {booking['table_id']}")
    print(f"Status: {booking['booking_status']}")
    print("--------------------------------")
    print(
        f"Save your booking ID: {booking['booking_id']}\n"
        "You need this ID if you want to cancel."
    )
    print("================================")

### Cell 13 - main_menu() // new bokningsflöde


In [ ]:
def booking_flow(tables, bookings):
    try:
        guest_count = int(
            input("\nHow many guests are you booking for? ")
        )

        if guest_count <= 0:
            print("The number of guests must be greater than zero.")
            return

    except ValueError:
        print("Please enter the number of guests using digits.")
        return

    booking_date = select_booking_date()
    booking_time = select_booking_time()

    selected_datetime = datetime.strptime(
        f"{booking_date} {booking_time}",
        "%Y-%m-%d %H:%M"
    )

    if selected_datetime <= datetime.now():
        print("The selected booking time has already passed.")
        return

    available_tables = find_available_tables(
        tables,
        bookings,
        guest_count,
        booking_date,
        booking_time
    )

    if not available_tables:
        print(
            "\nUnfortunately, no suitable tables are available "
            "at the selected date and time."
        )
        return

    customer_preference = input(
        "\nDescribe your preferred table, for example "
        "'window', 'quiet area' or 'terrace': "
    ).strip()

    if not customer_preference:
        customer_preference = "No special preference"

    recommendation = recommend_table_with_grok(
        available_tables,
        customer_preference
    )

    if recommendation is not None:
        print("\nAI recommendation:")
        print(recommendation)

    print("\nAvailable tables:")

    for table in available_tables:
        print(table.display_information())

    try:
        selected_table_id = int(
            input("\nEnter the table ID you want to book: ")
        )

    except ValueError:
        print("The table ID must be a number.")
        return

    selected_table = None

    for table in available_tables:
        if table.table_id == selected_table_id:
            selected_table = table
            break

    if selected_table is None:
        print("The selected table is not available.")
        return

    customer_name = input(
        "Enter the name for the booking: "
    ).strip()

    if not customer_name:
        print("The customer name cannot be empty.")
        return

    new_booking = create_booking(
        bookings,
        selected_table,
        customer_name,
        booking_date,
        booking_time,
        guest_count,
        "bookings.csv"
    )

    if new_booking is None:
        print("The booking could not be completed.")
        return

    display_booking_confirmation(new_booking)

### menu

In [ ]:
def main_menu():
    tables = load_tables("tables.csv")
    bookings = load_bookings("bookings.csv")

    if not tables:
        print("No tables could be loaded.")
        return

    while True:
        print("\n=== TABLEMATE ===")
        print("1. Book a table")
        print("2. Cancel a booking")
        print("3. Exit")

        menu_choice = input(
            "Choose an option (1-3): "
        ).strip()

        if menu_choice == "1":
            booking_flow(tables, bookings)

        elif menu_choice == "2":
            try:
                booking_id = int(
                    input("Enter the booking ID to cancel: ")
                )

                cancel_booking(
                    bookings,
                    booking_id,
                    "bookings.csv"
                )

            except ValueError:
                print("The booking ID must be a number.")

        elif menu_choice == "3":
            print("Thank you for using TableMate.")
            break

        else:
            print("Please choose a number between 1 and 3.")

In [ ]:
main_menu()